# LaLonde: propensity score có tìm lại được kết quả thí nghiệm không?

Chương trình đào tạo nghề NSW (Mỹ, 1975–1977) **chia ngẫu nhiên** người tham gia vào nhóm được đào
tạo (`treat = 1`) và nhóm không. Vì có random, chênh lệch thu nhập năm 1978 (`re78`) giữa hai nhóm là
tác động thật: con số chuẩn để đối chiếu.

LaLonde (1986) đặt câu hỏi: nếu **không có** nhóm control ngẫu nhiên, phải lấy người không tham gia
từ một khảo sát dân số (PSID) làm nhóm so sánh, thì các phương pháp quan sát có tìm lại được con
số đó không? Bộ `lalonde` gồm:

- `nsw_dw.dta`: mẫu thí nghiệm Dehejia–Wahba, 185 treated + 260 control ngẫu nhiên.
- `psid_controls.dta`: 2 490 người từ PSID, không ai tham gia chương trình.

Cột: `age`, `education`, `black`, `hispanic`, `married`, `nodegree`, `re74`, `re75` (thu nhập
trước chương trình, USD), `re78` (kết quả).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm hoặc hằng. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, vẽ, gọi hàm. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/06-causal-inference/exercises/intermediate/01-lalonde-propensity -q`

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

COVARIATES = ["age", "education", "black", "hispanic", "married", "nodegree", "re74", "re75"]

In [ ]:
# explore: đọc dữ liệu thật (tải bằng: uv run python data/download_dataset.py lalonde)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
nsw = pd.read_stata(ROOT / "data/raw/lalonde/nsw_dw.dta")
psid = pd.read_stata(ROOT / "data/raw/lalonde/psid_controls.dta")
# Bộ quan sát: người được đào tạo trong NSW + người không tham gia lấy từ PSID
obs = pd.concat([nsw[nsw.treat == 1], psid], ignore_index=True)
print(nsw.treat.value_counts().to_dict(), len(psid), len(obs))

## 1. Con số chuẩn và con số ngây thơ

In [ ]:
def diff_in_means(df: pd.DataFrame, outcome: str = "re78", treat: str = "treat") -> float:
    """mean(outcome | treat = 1) − mean(outcome | treat = 0)."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: thí nghiệm (chuẩn) so với so sánh ngây thơ với PSID
print("Thí nghiệm NSW:", round(diff_in_means(nsw), 2))
print("NSW treated vs PSID:", round(diff_in_means(obs), 2))

## 2. Balance: hai nhóm khác nhau cỡ nào?

**Standardized mean difference** (SMD) của một covariate:

$$
\mathrm{SMD} = \frac{\bar x_T - \bar x_C}{\sqrt{(s_T^2 + s_C^2)/2}}
$$

Quy ước thực hành: |SMD| < 0.1 là cân bằng. Khi có trọng số, tử số dùng trung bình **có trọng số**,
mẫu số giữ độ lệch chuẩn **không trọng số** (để so được trước và sau khi cân).

In [ ]:
def smd(df: pd.DataFrame, covariates: list[str], treat: str = "treat",
        weights: np.ndarray | None = None) -> pd.Series:
    """SMD của từng covariate (index là tên covariate). Phương sai dùng ddof=1, không trọng số."""
    # TODO: weights None nghĩa là mọi trọng số bằng 1; np.average(x, weights=w) cho trung bình có trọng số
    raise NotImplementedError

In [ ]:
# explore: thí nghiệm cân bằng, bộ quan sát thì không
pd.DataFrame({"NSW (random)": smd(nsw, COVARIATES), "NSW vs PSID": smd(obs, COVARIATES)}).round(2)

## 3. Propensity score và overlap

Propensity score $$e(x) = P(T = 1 \mid X = x)$$. Dùng logistic regression **không regularization**
(`statsmodels` `Logit` có hằng số), vì mục tiêu là cân bằng covariate chứ không phải dự đoán tốt
trên dữ liệu mới.

In [ ]:
def propensity_scores(df: pd.DataFrame, covariates: list[str], treat: str = "treat") -> np.ndarray:
    """P(treat = 1 | covariates) từ logistic regression có hằng số, không regularization."""
    # TODO: sm.Logit(y, sm.add_constant(X)).fit(disp=0), rồi predict
    raise NotImplementedError

In [ ]:
# explore: overlap. Phần lớn người PSID có propensity gần 0: họ không giống người tham gia chút nào
import matplotlib.pyplot as plt

ps = propensity_scores(obs, COVARIATES)
t = obs.treat.to_numpy() == 1
print("PSID có ps < 0.01:", int((ps[~t] < 0.01).sum()), "trên", int((~t).sum()))
plt.hist([ps[t], ps[~t]], bins=40, label=["NSW treated", "PSID"], log=True)
plt.xlabel("propensity score"); plt.legend(); plt.show()

## 4. IPW cho ATT

Câu hỏi của chương trình là tác động **lên người đã tham gia** (ATT). Trọng số ATT: người treated
giữ trọng số 1; người control nhận $$e(x)/(1 - e(x))$$, tức "người control này giống người treated
cỡ nào". Ước lượng chuẩn hoá (Hájek):

$$
\widehat{\mathrm{ATT}} = \bar y_T - \frac{\sum_{i \in C} w_i y_i}{\sum_{i \in C} w_i}
$$

Effective sample size của một nhóm có trọng số: $$\mathrm{ESS} = (\sum w_i)^2 / \sum w_i^2$$.

In [ ]:
def att_weights(ps: np.ndarray, treated: np.ndarray) -> np.ndarray:
    """1 cho treated, ps / (1 − ps) cho control."""
    # TODO
    raise NotImplementedError


def ipw_att(df: pd.DataFrame, ps: np.ndarray, outcome: str = "re78", treat: str = "treat") -> float:
    """ATT bằng IPW chuẩn hoá (Hájek)."""
    # TODO
    raise NotImplementedError


def effective_sample_size(weights: np.ndarray) -> float:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: IPW, balance sau khi cân, và cái giá: chỉ vài chục người PSID gánh toàn bộ nhóm control
w = att_weights(ps, obs.treat.to_numpy())
print("IPW ATT:", round(ipw_att(obs, ps), 1))
print("ESS control:", round(effective_sample_size(w[~t]), 1), "trên", int((~t).sum()))
print("Trọng số control lớn nhất chiếm", round(w[~t].max() / w[~t].sum(), 3), "tổng trọng số")
pd.DataFrame({"trước": smd(obs, COVARIATES), "sau IPW": smd(obs, COVARIATES, weights=w)}).round(2)

## 5. Matching

1-nearest-neighbor matching trên propensity score, **có hoàn lại** (một người control được ghép cho
nhiều người treated): mỗi người treated lấy người control có `|ps_c − ps_t|` nhỏ nhất. ATT là trung
bình của `y_t − y_c(ghép)`. Hoà thì lấy người control xuất hiện trước trong `df`.

In [ ]:
def nn_match_att(df: pd.DataFrame, ps: np.ndarray, outcome: str = "re78", treat: str = "treat") -> float:
    """ATT bằng 1-NN matching trên propensity score, có hoàn lại."""
    # TODO: ma trận khoảng cách |ps_t[:, None] − ps_c[None, :]|, argmin theo cột
    raise NotImplementedError

In [ ]:
# explore: matching dùng bao nhiêu người control khác nhau?
print("NN matching ATT:", round(nn_match_att(obs, ps), 1))
match = np.abs(ps[t][:, None] - ps[~t][None, :]).argmin(axis=1)
print("Số người PSID khác nhau được ghép:", len(set(match)))

## 6. Độ nhạy theo đặc tả và trimming

Thêm `age²`, `education²` và hai biến "không có thu nhập năm 74 / 75" (đặc tả của Dehejia–Wahba)
vào mô hình propensity. Rồi thử bỏ những người có propensity ngoài khoảng [0.05, 0.95].
Ước lượng thay đổi bao nhiêu? Điều đó nói gì về độ tin cậy của một con số IPW duy nhất?

In [ ]:
# explore: đặc tả khác, trimming
obs2 = obs.assign(age2=obs.age**2, educ2=obs.education**2,
                  u74=(obs.re74 == 0).astype(float), u75=(obs.re75 == 0).astype(float))
ps_dw = propensity_scores(obs2, COVARIATES + ["age2", "educ2", "u74", "u75"])
print("IPW, đặc tả Dehejia–Wahba:", round(ipw_att(obs2, ps_dw), 1))
print("NN matching, đặc tả Dehejia–Wahba:", round(nn_match_att(obs2, ps_dw), 1))
for name, p in [("8 covariate", ps), ("Dehejia–Wahba", ps_dw)]:
    keep = (p > 0.05) & (p < 0.95)
    print(name, "trim [0.05, 0.95]:", round(ipw_att(obs[keep], p[keep]), 1), "| n =", int(keep.sum()))

## 7. Kết luận

Viết 4–6 câu: IPW và matching so với con số thí nghiệm thế nào; vì sao con số ngây thơ sai tới mức
đổi dấu; overlap và ESS nói gì; nếu không có thí nghiệm để đối chiếu thì bạn tin ước lượng nào,
với mức độ chắc chắn nào.